In [ ]:
# Cell 1 — Bootstrap：定位项目根目录 + 输出目录 + 自动定位最新 data_03

import os
import glob
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np

def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    markers = [".git", "pyproject.toml", "requirements.txt", "configs", "models", "Data", "src", "docs"]
    for p in [start, *start.parents]:
        if any((p / m).exists() for m in markers):
            return p
    return start

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "Data"
print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_DIR =", DATA_DIR)

# 1) output dir
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = DATA_DIR / f"data_07_thread_stance_nli_{timestamp}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def out(name: str) -> str:
    return str(OUTPUT_DIR / name)

print("OUTPUT_DIR =", OUTPUT_DIR)

# 2) locate latest data_03_*
candidates = sorted(glob.glob(str(DATA_DIR / "data_03_*")), reverse=True)
if not candidates:
    raise FileNotFoundError(f"❌ No folder found: {DATA_DIR}/data_03_*")

LATEST_03 = Path(candidates[0])
print("LATEST_03 =", LATEST_03)

TOP_PARQUET = str(LATEST_03 / "cleaned_top_level_comments.parquet")
REPLY_PARQUET = str(LATEST_03 / "cleaned_replies.parquet")

if not os.path.exists(TOP_PARQUET):
    raise FileNotFoundError(f"❌ Missing TOP_PARQUET: {TOP_PARQUET}")
if not os.path.exists(REPLY_PARQUET):
    raise FileNotFoundError(f"❌ Missing REPLY_PARQUET: {REPLY_PARQUET}")

print("TOP_PARQUET =", TOP_PARQUET)
print("REPLY_PARQUET =", REPLY_PARQUET)


In [ ]:
# Cell 2 — Load & Normalize：读取 parents/replies + 统一字段

top = pd.read_parquet(TOP_PARQUET)
replies = pd.read_parquet(REPLY_PARQUET)

print("top shape:", top.shape)
print("replies shape:", replies.shape)

def unify_text(df_):
    df_ = df_.copy()
    if "text_model" in df_.columns:
        df_ = df_.rename(columns={"text_model": "text"})
    elif "text_clean" in df_.columns:
        df_ = df_.rename(columns={"text_clean": "text"})
    elif "text" not in df_.columns:
        raise KeyError("❌ Missing text column: text_model/text_clean/text")
    df_["text"] = df_["text"].astype(str).str.strip()
    return df_

top = unify_text(top)
replies = unify_text(replies)

# required columns
for c in ["thread_id", "cell_id", "text"]:
    if c not in top.columns:
        raise KeyError(f"❌ top missing col: {c}. got: {list(top.columns)}")

if "thread_id" not in replies.columns:
    raise KeyError(f"❌ replies missing col: thread_id. got: {list(replies.columns)}")

top["thread_id"] = top["thread_id"].astype(str)
replies["thread_id"] = replies["thread_id"].astype(str)

# likes columns (optional)
TOP_LIKE_COL = "like_count_parent" if "like_count_parent" in top.columns else ("like_count" if "like_count" in top.columns else None)
REP_LIKE_COL = "like_count_reply" if "like_count_reply" in replies.columns else ("like_count" if "like_count" in replies.columns else None)

print("TOP_LIKE_COL:", TOP_LIKE_COL)
print("REP_LIKE_COL:", REP_LIKE_COL)


In [ ]:
# Cell 3 — (可选) 过滤策略：只做 labor 语料、过滤超短母评论

ONLY_LABOR = True
MIN_PARENT_TOKENS = 5

parent_df = top.copy()
parent_df["token_count"] = parent_df["text"].str.split().str.len()

if MIN_PARENT_TOKENS is not None:
    parent_df = parent_df[parent_df["token_count"] >= MIN_PARENT_TOKENS].copy()

if ONLY_LABOR:
    parent_df = parent_df[parent_df["cell_id"].str.contains("labor", na=False, case=False)].copy()

reply_df = replies.dropna(subset=["thread_id", "text"]).copy()
reply_df["token_count"] = reply_df["text"].str.split().str.len()

print("parent_df:", parent_df.shape, "| unique threads:", parent_df["thread_id"].nunique())
print("reply_df:", reply_df.shape, "| unique threads:", reply_df["thread_id"].nunique())


In [ ]:
# Cell 4 — Build pairs：reply join parent（thread）

parent_map = parent_df[["thread_id", "text", "cell_id"]].rename(columns={"text": "parent_text"}).drop_duplicates("thread_id")

pairs = reply_df.merge(parent_map, on="thread_id", how="inner")
print("pairs shape:", pairs.shape)

if len(pairs) == 0:
    raise RuntimeError("❌ pairs is empty after join. Check thread_id consistency or filters.")


In [ ]:
# Cell 5 — Load LOCAL NLI model：本地加载（不联网）

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

NLI_MODEL_DIR = PROJECT_ROOT / "models" / "nli_deberta_mnli_fever_anli"
if not NLI_MODEL_DIR.exists():
    raise FileNotFoundError(f"❌ NLI model dir not found: {NLI_MODEL_DIR}. Run 00_setup_models.ipynb first.")

device = 0 if torch.cuda.is_available() else -1
print("device:", device, "(0 means GPU, -1 means CPU)")

tok = AutoTokenizer.from_pretrained(str(NLI_MODEL_DIR), local_files_only=True)
mdl = AutoModelForSequenceClassification.from_pretrained(str(NLI_MODEL_DIR), local_files_only=True)

nli = pipeline(
    "text-classification",
    model=mdl,
    tokenizer=tok,
    device=device,
    truncation=True
)


In [10]:
# Cell 6 — Inference：tqdm 进度条 + max_length 提速 + 缓存

from tqdm.auto import tqdm
from math import ceil
import time

# cache
CACHE_PATH = Path(out("reply_stance_cache.parquet"))

MAX_LEN = 192    # 可改 192 更快
BS = 32 if device != -1 else 16

def clip_text(s, max_chars):
    return str(s)[:max_chars]

# 字符截断（配合 max_length）
pairs["parent_clip"] = pairs["parent_text"].apply(lambda x: clip_text(x, 400))
pairs["reply_clip"]  = pairs["text"].apply(lambda x: clip_text(x, 250))

if CACHE_PATH.exists():
    print("✅ Loading cached reply stance:", CACHE_PATH)
    reply_stance_df = pd.read_parquet(CACHE_PATH)
else:
    inputs = [{"text": p, "text_pair": h} for p, h in zip(pairs["parent_clip"].tolist(), pairs["reply_clip"].tolist())]
    total_batches = ceil(len(inputs) / BS)

    out_labels, out_scores = [], []
    print(f"⏳ Running NLI inference... pairs={len(inputs)}, BS={BS}, max_length={MAX_LEN}")
    t0 = time.time()

    def batched(lst, bs):
        for i in range(0, len(lst), bs):
            yield lst[i:i+bs]

    for chunk in tqdm(batched(inputs, BS), total=total_batches):
        preds = nli(chunk, truncation=True, max_length=MAX_LEN)
        out_labels.extend([x["label"] for x in preds])
        out_scores.extend([float(x["score"]) for x in preds])

    print(f"✅ NLI inference done. elapsed={time.time()-t0:.1f}s")

    pairs["nli_label_raw"] = out_labels
    pairs["nli_score"] = out_scores

    def normalize(lbl: str) -> str:
        lbl = (lbl or "").lower()
        if "entail" in lbl:
            return "support"
        if "contrad" in lbl:
            return "oppose"
        return "neutral"

    pairs["stance"] = pairs["nli_label_raw"].apply(normalize)

    # weight = reply likes + 1
    if REP_LIKE_COL is not None and REP_LIKE_COL in pairs.columns:
        pairs["w"] = pairs[REP_LIKE_COL].fillna(0).astype(float) + 1.0
    else:
        pairs["w"] = 1.0
    
    # keep minimal columns (avoid super large file) — robust column selection
    def pick_col(df_, candidates):
        for c in candidates:
            if c in df_.columns:
                return c
        return None

    cell_col = pick_col(pairs, ["cell_id", "cell_id_x", "cell_id_y"])
    reply_text_col = pick_col(pairs, ["text", "reply_text"])  # reply text column in pairs is usually "text"

    cols = ["thread_id", "stance", "nli_score", "w", "parent_text"]
    if cell_col is not None:
        cols.append(cell_col)
    if reply_text_col is not None:
        cols.append(reply_text_col)

    missing = [c for c in cols if c not in pairs.columns]
    if missing:
        raise KeyError(f"❌ Missing columns in pairs: {missing}. Available: {list(pairs.columns)}")

    reply_stance_df = pairs[cols].copy()
    if cell_col is not None and cell_col != "cell_id":
        reply_stance_df = reply_stance_df.rename(columns={cell_col: "cell_id"})
    if reply_text_col is not None and reply_text_col != "reply_text":
        reply_stance_df = reply_stance_df.rename(columns={reply_text_col: "reply_text"})

    reply_stance_df.to_parquet(CACHE_PATH, index=False)
    print("✅ Saved cache:", CACHE_PATH)

print("reply_stance_df:", reply_stance_df.shape)


⏳ Running NLI inference... pairs=23503, BS=16, max_length=192


  0%|          | 0/1469 [00:00<?, ?it/s]

✅ NLI inference done. elapsed=2160.4s
✅ Saved cache: D:\coding\projects\Python\Youtube Comment Sentiment Analysis\Data\data_07_thread_stance_nli_20260301_010211\reply_stance_cache.parquet
reply_stance_df: (23503, 7)


In [12]:
# Cell 7 — Aggregate：thread 级共鸣/争议 + thread_panel（带母评论指标）

# thread aggregation
thread_agg = (
    reply_stance_df
    .groupby(["thread_id", "stance"])["w"]
    .sum()
    .reset_index()
    .pivot_table(index=["thread_id"], columns="stance", values="w", fill_value=0.0)
    .reset_index()
)

for c in ["support", "oppose", "neutral"]:
    if c not in thread_agg.columns:
        thread_agg[c] = 0.0

thread_agg["w_total"] = thread_agg[["support", "oppose", "neutral"]].sum(axis=1)
thread_agg["support_rate"] = thread_agg["support"] / (thread_agg["w_total"] + 1e-9)
thread_agg["oppose_rate"]  = thread_agg["oppose"]  / (thread_agg["w_total"] + 1e-9)
thread_agg["neutral_rate"] = thread_agg["neutral"] / (thread_agg["w_total"] + 1e-9)

# controversy/resonance definition
thread_agg["controversy"] = 1.0 - np.abs(thread_agg["support_rate"] - thread_agg["oppose_rate"])
thread_agg["resonance"] = thread_agg["support_rate"]

thread_stance_df = thread_agg.copy()
print("thread_stance_df:", thread_stance_df.shape)

# thread panel: merge parent meta (likes/reply_count if exist)
panel_cols = ["thread_id", "cell_id", "text"]  # ✅ parent_df 里是 text，不是 parent_text
if TOP_LIKE_COL is not None and TOP_LIKE_COL in parent_df.columns:
    panel_cols.append(TOP_LIKE_COL)
if "total_reply_count" in parent_df.columns:
    panel_cols.append("total_reply_count")

# ✅ 从 text 改名为 parent_text
thread_panel = (
    parent_df[panel_cols]
    .rename(columns={"text": "parent_text"})
    .drop_duplicates("thread_id")
)

thread_panel = thread_panel.merge(thread_stance_df, on="thread_id", how="left")

thread_panel = thread_panel.merge(thread_stance_df, on="thread_id", how="left")

# fill threads with no replies (if any)
for c in ["support", "oppose", "neutral", "w_total", "support_rate", "oppose_rate", "neutral_rate", "controversy", "resonance"]:
    if c in thread_panel.columns:
        thread_panel[c] = thread_panel[c].fillna(0.0)

print("thread_panel:", thread_panel.shape)


thread_stance_df: (2604, 10)
thread_panel: (22520, 23)


In [13]:
# Cell 8 — (可选) topic_stance 占位：给 08 以后 merge 用

topic_stance_df = pd.DataFrame(columns=[
    "topic_id", "support", "oppose", "neutral", "w_total",
    "support_rate", "oppose_rate", "neutral_rate", "controversy", "resonance"
])
print("topic_stance_df placeholder ready.")

topic_stance_df placeholder ready.


In [15]:
# Cell 9 — Save outputs：保存 CSV（用于 08 & dashboard）

OUT_REPLY = out("reply_stance.csv")
OUT_THREAD = out("thread_stance.csv")
OUT_PANEL = out("thread_panel_stance.csv")
OUT_TOPIC = out("topic_stance.csv")

reply_stance_df.to_csv(OUT_REPLY, index=False, encoding="utf-8-sig")
thread_stance_df.to_csv(OUT_THREAD, index=False, encoding="utf-8-sig")
thread_panel.to_csv(OUT_PANEL, index=False, encoding="utf-8-sig")
topic_stance_df.to_csv(OUT_TOPIC, index=False, encoding="utf-8-sig")

print("✅ Saved:")
print(" -", OUT_REPLY)
print(" -", OUT_THREAD)
print(" -", OUT_PANEL)
print(" -", OUT_TOPIC)


✅ Saved:
 - D:\coding\projects\Python\Youtube Comment Sentiment Analysis\Data\data_07_thread_stance_nli_20260301_010211\reply_stance.csv
 - D:\coding\projects\Python\Youtube Comment Sentiment Analysis\Data\data_07_thread_stance_nli_20260301_010211\thread_stance.csv
 - D:\coding\projects\Python\Youtube Comment Sentiment Analysis\Data\data_07_thread_stance_nli_20260301_010211\thread_panel_stance.csv
 - D:\coding\projects\Python\Youtube Comment Sentiment Analysis\Data\data_07_thread_stance_nli_20260301_010211\topic_stance.csv


In [1]:
# Cell 10 — Generate NLI Stance Visualization (Load from existing CSV, 0 算力消耗)
# -------------------------------------------------------------------
import os
import glob
import pandas as pd
import plotly.express as px

# 1. 自动寻找最新的 07 模块输出文件夹
try:
    from pyprojroot import here
    PROJECT_ROOT = here()
except ImportError:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

DATA_DIR = os.path.join(PROJECT_ROOT, "Data")
cands = glob.glob(os.path.join(DATA_DIR, "data_07_thread_stance_nli*"))
dirs = [d for d in cands if os.path.isdir(d)]

if not dirs:
    raise FileNotFoundError("❌ 找不到 Data/data_07_thread_stance_nli* 文件夹，请确认你已经跑过 NLI 模型！")

latest_07_dir = max(dirs, key=os.path.getmtime)
print(f"📂 锁定最新 07 文件夹: {latest_07_dir}")

# 2. 读取之前跑完保存的 CSV 数据
panel_csv = os.path.join(latest_07_dir, "thread_panel_stance.csv")
if not os.path.exists(panel_csv):
    raise FileNotFoundError(f"❌ 找不到数据文件: {panel_csv}，请确保前面的 Cell 9 已经将其导出。")

thread_panel = pd.read_csv(panel_csv)
print(f"✅ 成功加载 NLI 历史数据，包含 {len(thread_panel)} 条 Thread 记录。")

# 🌟 核心修复1：处理 Pandas merge 产生的 _x 后缀问题
if "support_x" in thread_panel.columns:
    thread_panel = thread_panel.rename(columns={
        "support_x": "support", 
        "oppose_x": "oppose", 
        "neutral_x": "neutral"
    })

# 🌟 核心修复2：业务化组别名称映射，保持与06/08的全局统一
GROUP_LABEL_MAP = {
    'shein_labor': 'SHEIN / Labor', 
    'shein_environment': 'SHEIN / Environment',
    'non_shein_labor': 'Peers / Labor', 
    'non_shein_environment': 'Peers / Environment'
}

# 3. 聚合各阵营的立场总热度
viz_stance = thread_panel.groupby("cell_id")[["support", "oppose", "neutral"]].sum().reset_index()

# 将内部的 cell_id 映射为正式的 Group 标签
viz_stance["Group"] = viz_stance["cell_id"].map(GROUP_LABEL_MAP).fillna(viz_stance["cell_id"])

# 4. 计算百分比，做成 100% 堆叠条形图
viz_stance["total"] = viz_stance[["support", "oppose", "neutral"]].sum(axis=1) + 1e-9
# 🌟 直接将其重命名为首字母大写的正式指标名
viz_stance["Support"] = viz_stance["support"] / viz_stance["total"] * 100
viz_stance["Oppose"] = viz_stance["oppose"] / viz_stance["total"] * 100
viz_stance["Neutral"] = viz_stance["neutral"] / viz_stance["total"] * 100

# 5. 融化数据适应 Plotly，剔除掉旧的小写列和 cell_id
viz_melt = viz_stance.drop(columns=["support", "oppose", "neutral", "total", "cell_id"]).melt(
    id_vars="Group", var_name="Stance", value_name="Percentage (%)"
)

# 🌟 核心修复3：强制设定分类的顺序（支持 -> 中立 -> 反对），保证堆叠图视觉逻辑最顺畅
viz_melt["Stance"] = pd.Categorical(viz_melt["Stance"], categories=["Support", "Neutral", "Oppose"], ordered=True)
viz_melt = viz_melt.sort_values("Stance")

# 6. 绘制专业视角的横向对比条形图
fig_stance = px.bar(
    viz_melt, 
    x="Percentage (%)", 
    y="Group", 
    color="Stance",
    orientation="h", 
    color_discrete_map={
        "Support": "#00CC96",   # 薄荷绿
        "Neutral": "#D3D3D3",   # 浅灰 (用灰色代表中立最合适)
        "Oppose": "#EF553B"     # 橘红
    },
    # 🌟 核心修复4：去情绪化的标准标题，加上解释共鸣度与争议度的副标题
    title="Inferred Stance Distribution by Group (NLI)<br><sup>Thread-level resonance is defined as support rate; controversy increases as support and oppose become more balanced.</sup>"
)

fig_stance.update_layout(
    # 🌟 核心修复5：将 X 轴正名为 Weighted Stance Share
    xaxis_title="Weighted Stance Share (%)", 
    yaxis_title="", 
    height=450,
    margin=dict(l=40, r=40, t=80, b=40), # 加大 t 给副标题留出足够空间
    legend_title="Stance",
    plot_bgcolor="#f9f9f9", paper_bgcolor="#ffffff"
)

# 🌟 核心修复6：定制化极简 Hover 提示框
fig_stance.update_traces(
    hovertemplate="<b>Group:</b> %{y}<br><b>Stance:</b> %{data.name}<br><b>Share:</b> %{x:.1f}%<extra></extra>"
)

# 7. 导出 HTML 资产到同一个 07 文件夹里
OUT_HTML = os.path.join(latest_07_dir, "06_nli_stance_distribution_stacked.html")
fig_stance.write_html(OUT_HTML, include_plotlyjs="cdn", full_html=True)
fig_stance.show()

print(f"✅ NLI Stance 宏观大盘图已生成，专门投喂给 Dashboard Tab 1: {OUT_HTML}")

📂 锁定最新 07 文件夹: d:\coding\projects\Python\Youtube Comment Sentiment Analysis\Data\data_07_thread_stance_nli_20260301_010211
✅ 成功加载 NLI 历史数据，包含 22520 条 Thread 记录。


✅ NLI Stance 宏观大盘图已生成，专门投喂给 Dashboard Tab 1: d:\coding\projects\Python\Youtube Comment Sentiment Analysis\Data\data_07_thread_stance_nli_20260301_010211\06_nli_stance_distribution_stacked.html
